# Real-Time Audio Processing Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: ring buffer

In [ ]:
```python

import collections

class RingBuffer:

    def __init__(self, capacity):

        self.buf = collections.deque(maxlen=capacity)

    def write(self, frame):

        self.buf.extend(frame)

    def read(self, n):

        return [self.buf.popleft() for _ in range(min(n, len(self.buf)))]

    def level(self):

        return len(self.buf)

In [ ]:
```

Capacity determines max buffering latency. 32,000 samples at 16 kHz = 2 s.

### Step 2: VAD gate

In [ ]:
```python

def simple_energy_vad(frame, threshold=0.01):

    return sum(x * x for x in frame) / len(frame) > threshold ** 2

In [ ]:
```

Replace with Silero VAD in production:

In [ ]:
```python

import torch

vad, _ = torch.hub.load("snakers4/silero-vad", "silero_vad")

is_speech = vad(torch.tensor(frame), 16000).item() > 0.5

In [ ]:
```

### Step 3: streaming ASR

In [ ]:
```python

# Parakeet-CTC-0.6B streaming via NeMo

from nemo.collections.asr.models import EncDecCTCModelBPE

asr = EncDecCTCModelBPE.from_pretrained("nvidia/parakeet-ctc-0.6b")

# chunk_ms=320 ms, look_ahead_ms=80 ms

for chunk in audio_stream():

    partial_text = asr.transcribe_streaming(chunk)

    print(partial_text, end="\r")

In [ ]:
```

### Step 4: interruption handler

In [ ]:
```python

class Dialog:

    def __init__(self):

        self.tts_task = None

    def on_user_speech(self, frame):

        if self.tts_task and not self.tts_task.done():

            self.tts_task.cancel()   # barge-in

        # then feed to streaming ASR

    def on_final_user_utterance(self, text):

        self.tts_task = asyncio.create_task(self.reply(text))

    async def reply(self, text):

        async for tts_chunk in llm_then_tts(text):

            speaker.write(tts_chunk)

In [ ]:
```

Hinges on async I/O and cancellable TTS streaming. WebRTC peerconnection.stop() on the audio track is the canonical way.

## Exercises

In [ ]:
1. **Easy.** Run `code/main.py`. Simulates a ring buffer + energy VAD; prints stage latencies for a fake 10-second stream.
2. **Medium.** Using `sounddevice`, build a passthrough loop that processes your mic in 20 ms frames and prints VAD state at each frame.
3. **Hard.** Build a full duplex echo test with `aiortc`: browser → WebRTC → Python → WebRTC → browser. Measure glass-to-glass latency with a 1 kHz pulse.